<a href="https://colab.research.google.com/github/antara2002nigudkar/ML-safety-file/blob/main/Exercise_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/Colab Notebooks"

'Exercise 3 ML_Safety.ipynb'   ML_Safety.ipynb	  test.zip
'Exercise 4 ML_Safety.ipynb'   test-fog.zip	  train.zip
'Exercise 5 ML_Safety.ipynb'   test-night.zip	  validation.zip
'Exercise 7.ipynb'	       test-town-01.zip


In [3]:
!unzip -q "/content/drive/MyDrive/Colab Notebooks/validation.zip" -d "/content/validation"

In [4]:
!ls "/content/validation/validation"

actions.feather     gnss.feather  labels.feather      sim.log
carla.log	    imu.feather   rgb-front	      weather.feather
collisions.feather  labels.csv	  segmentation-front


In [5]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

In [6]:
class CARLADataset(Dataset):

    def __init__(self, root_dir):

        self.root_dir = root_dir
        self.labels = pd.read_csv(
            os.path.join(root_dir, "labels.csv")
        )

        self.rgb_dir = os.path.join(
            root_dir,
            "rgb-front"
        )

        self.transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        img_name = str(
            int(row["frame"])
        ).zfill(6)

        img_path = os.path.join(
            self.rgb_dir,
            f"{img_name}.jpg"
        )

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        label = torch.tensor([
            row["has_traffic_light"],
            row["has_pedestrian"],
            row["has_vehicle"]
        ], dtype=torch.float32)

        return image, label

In [7]:
val_path = "/content/validation/validation"

val_dataset = CARLADataset(val_path)

print("Dataset size:", len(val_dataset))

Dataset size: 3600


In [8]:
img, label = val_dataset[0]

print(img.shape)
print(label)

torch.Size([3, 224, 224])
tensor([0., 1., 0.])


In [9]:
import torchvision.models as models
import torch.nn as nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [10]:
def get_model():

    model = models.resnet18(weights=None)

    model.fc = nn.Linear(512, 1)

    return model

In [11]:
ped_model = get_model().to(device)
traffic_model = get_model().to(device)
vehicle_model = get_model().to(device)

ped_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/ped_model.pth",
        map_location=device
    )
)

traffic_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/traffic_model.pth",
        map_location=device
    )
)

vehicle_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/vehicle_model.pth",
        map_location=device
    )
)

ped_model.eval()
traffic_model.eval()
vehicle_model.eval()

print("Models loaded")

Models loaded


In [12]:
loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

imgs, labels = next(iter(loader))

with torch.no_grad():

    out = ped_model(
        imgs.to(device)
    )

print("Output shape:", out.shape)
print("Label shape:", labels.shape)

Output shape: torch.Size([8, 1])
Label shape: torch.Size([8, 3])


In [13]:
label_map = {
    "traffic": 0,
    "pedestrian": 1,
    "vehicle": 2
}

In [14]:
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [20]:
import torch.nn.functional as F

def eval_nll(model, loader, T, label_idx):

    model.eval()

    total_loss = 0
    total_samples = 0

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)

            target = labels[:, label_idx].float().unsqueeze(1).to(device)

            logits = model(imgs)

            logits = logits / T

            loss = F.binary_cross_entropy_with_logits(
                logits,
                target,
                reduction="sum"
            )

            total_loss += loss.item()
            total_samples += imgs.size(0)

    return total_loss / total_samples

In [ ]:
print(
    eval_nll(
        ped_model,
        val_loader,
        1.0,
        label_map["pedestrian"]
    )
)

0.8094668153921764


In [21]:
def find_best_T(model, loader, label_idx):

    best_T = 1.0
    best_nll = float("inf")

    for T in [round(x * 0.1, 1) for x in range(5, 31)]:

        nll = eval_nll(model, loader, T, label_idx)

        if nll < best_nll:
            best_nll = nll
            best_T = T

    return best_T, best_nll

In [23]:
label_map = {
    "traffic": 0,
    "pedestrian": 1,
    "vehicle": 2
}

best_Ts = {}

for name, model in {
    "pedestrian": ped_model,
    "traffic": traffic_model,
    "vehicle": vehicle_model
}.items():

    T, nll = find_best_T(model, val_loader, label_map[name])

    best_Ts[name] = T

    print(f"{name}: T = {T}, NLL = {nll:.4f}")

pedestrian: T = 3.0, NLL = 0.5665
traffic: T = 0.5, NLL = 0.6630
vehicle: T = 1.4, NLL = 0.6429


In [15]:
import torch
import numpy as np

def get_probs(model, loader, T, label_idx):

    model.eval()

    probs_list = []
    labels_list = []

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)

            logits = model(imgs)

            logits = logits / T

            probs = torch.sigmoid(logits)

            probs_list.append(probs.cpu())
            labels_list.append(labels[:, label_idx].unsqueeze(1))

    return torch.cat(probs_list), torch.cat(labels_list)

In [16]:
def compute_ece(probs, labels, n_bins=10):

    probs = probs.numpy().flatten()
    labels = labels.numpy().flatten()

    bin_edges = np.linspace(0, 1, n_bins + 1)

    ece = 0.0

    for i in range(n_bins):

        mask = (probs > bin_edges[i]) & (probs <= bin_edges[i+1])

        if mask.sum() == 0:
            continue

        acc = labels[mask].mean()
        conf = probs[mask].mean()

        ece += abs(acc - conf) * (mask.sum() / len(probs))

    return ece

In [24]:
label_map = {
    "traffic": 0,
    "pedestrian": 1,
    "vehicle": 2
}

results = {}

for name, model in {
    "pedestrian": ped_model,
    "traffic": traffic_model,
    "vehicle": vehicle_model
}.items():

    idx = label_map[name]

    # BEFORE scaling
    probs_b, labels_b = get_probs(model, val_loader, 1.0, idx)
    ece_before = compute_ece(probs_b, labels_b)

    # AFTER scaling
    T = best_Ts[name]
    probs_a, labels_a = get_probs(model, val_loader, T, idx)
    ece_after = compute_ece(probs_a, labels_a)

    results[name] = (ece_before, ece_after, T)

    print(f"\n{name}")
    print(f"ECE before: {ece_before:.4f}")
    print(f"ECE after : {ece_after:.4f}")
    print(f"Best T    : {T}")


pedestrian
ECE before: 0.1551
ECE after : 0.0743
Best T    : 3.0

traffic
ECE before: 0.2370
ECE after : 0.2203
Best T    : 0.5

vehicle
ECE before: 0.0538
ECE after : 0.0018
Best T    : 1.4


In [25]:
CFN = 100
CFP = 1

In [26]:
import torch

def get_preds(model, loader, threshold):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)

            logits = model(imgs)

            probs = torch.sigmoid(logits)

            preds = (probs > threshold).float()

            all_preds.append(preds.cpu())
            all_labels.append(labels[:, 1].unsqueeze(1))  # pedestrian index

    return torch.cat(all_preds), torch.cat(all_labels)

In [27]:
def compute_cost(preds, labels):

    FN = ((preds == 0) & (labels == 1)).sum().item()
    FP = ((preds == 1) & (labels == 0)).sum().item()

    L = CFN * FN + CFP * FP

    return L, FN, FP

In [28]:
# τ = 0.5
pred_05, lab = get_preds(ped_model, val_loader, 0.5)
L_05, FN_05, FP_05 = compute_cost(pred_05, lab)

# τ* = 0.0099
pred_opt, lab = get_preds(ped_model, val_loader, 0.0099)
L_opt, FN_opt, FP_opt = compute_cost(pred_opt, lab)

In [29]:
print("=== τ = 0.5 ===")
print("FN:", FN_05, "FP:", FP_05)
print("Loss:", L_05)

print("\n=== τ* = 0.0099 ===")
print("FN:", FN_opt, "FP:", FP_opt)
print("Loss:", L_opt)

=== τ = 0.5 ===
FN: 725 FP: 218
Loss: 72718

=== τ* = 0.0099 ===
FN: 131 FP: 2056
Loss: 15156


In [30]:
def get_preds_temp(model, loader, threshold, T, label_idx):

    model.eval()

    preds_list = []
    labels_list = []

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)

            logits = model(imgs)

            probs = torch.sigmoid(logits / T)

            preds = (probs > threshold).float()

            preds_list.append(preds.cpu())
            labels_list.append(labels[:, label_idx].unsqueeze(1))

    return torch.cat(preds_list), torch.cat(labels_list)

In [31]:
def compute_cost(preds, labels):

    FN = ((preds == 0) & (labels == 1)).sum().item()
    FP = ((preds == 1) & (labels == 0)).sum().item()

    return 100 * FN + FP, FN, FP

In [32]:
T = best_Ts["pedestrian"]

# τ = 0.5
pred_05, lab = get_preds_temp(ped_model, val_loader, 0.5, T, label_map["pedestrian"])
L_05, FN_05, FP_05 = compute_cost(pred_05, lab)

# τ* = 0.0099
pred_opt, lab = get_preds_temp(ped_model, val_loader, 0.0099, T, label_map["pedestrian"])
L_opt, FN_opt, FP_opt = compute_cost(pred_opt, lab)

In [33]:
print("=== Temperature Scaled Model ===")

print("\nτ = 0.5")
print("FN:", FN_05, "FP:", FP_05)
print("Loss:", L_05)

print("\nτ* = 0.0099")
print("FN:", FN_opt, "FP:", FP_opt)
print("Loss:", L_opt)

=== Temperature Scaled Model ===

τ = 0.5
FN: 725 FP: 218
Loss: 72718

τ* = 0.0099
FN: 0 FP: 2686
Loss: 2686
